# EDA & Data Split
**목표:** page_features / response 데이터 품질 확인 → train / val / test split 생성

| 파일 | 행 수 | 설명 |
|------|-------|------|
| `page_features.csv` | 327 | 피케어드 상품 페이지 feature |
| `response.csv` | 6,000 | AI 3-way ranking 결과 (2 engines × 3,000 set_id) |

### 핵심 주의사항
> `brand_ko` ≠ unique 상품 식별자. 브랜드당 최대 20개 상품이 존재.
> URL이 실제 unique key. response.csv는 brand 이름으로만 SKU를 기록하므로,
> 브랜드에 상품이 여러 개 있을 경우 어느 상품을 평가했는지 **특정 불가**.
> → 모델 feature 구성 시 brand-level 집계 또는 추가 매핑 필요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings; warnings.filterwarnings('ignore')
import os, json

RAW_DIR   = '../data/raw'
PROC_DIR  = '../data/processed'
SPLIT_DIR = '../data/splits'

pf   = pd.read_csv(f'{RAW_DIR}/page_features.csv', encoding='utf-8')
resp = pd.read_csv(f'{RAW_DIR}/response.csv',      encoding='utf-8')

# 완전 빈 컬럼 제거
pf = pf.drop(columns=[c for c in pf.columns if pf[c].isna().all()])

print(f'page_features : {pf.shape}')
print(f'response      : {resp.shape}')
print(f'URL unique    : {pf["url"].nunique()} / {len(pf)} rows')
print(f'brand_ko unique: {pf["brand_ko"].nunique()} brands → multi-product 브랜드 존재')

---
## 1. page_features.csv 품질 확인
### 1-1. 결측치

In [ ]:
miss_pct = (pf.isnull().sum() / len(pf) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]
print(f'결측치 있는 컬럼: {len(miss_pct)}개')
display(miss_pct.rename('missing_%').to_frame())

### 1-2. Sentinel 값 처리 (-1 = "정보 없음")

In [ ]:
sentinel_cols = {
    'ph_value'              : 'pH 기재 없음',
    'aggregate_rating_value': '리뷰 점수 없음',
    'aggregate_rating_count': '리뷰 수 없음',
    'volume_ml'             : '용량 정보 없음',
}

rows_s = []
for col, desc in sentinel_cols.items():
    if col not in pf.columns:
        continue
    n = (pf[col] == -1).sum()
    valid = pf.loc[pf[col] != -1, col]
    rows_s.append({
        'column': col, 'description': desc,
        '-1 count': n, '-1 %': f'{n/len(pf)*100:.1f}%',
        'valid min': valid.min() if len(valid) else float('nan'),
        'valid max': valid.max() if len(valid) else float('nan'),
    })

display(pd.DataFrame(rows_s).set_index('column'))

pf_clean = pf.copy()
for col in sentinel_cols:
    if col in pf_clean.columns:
        pf_clean[col] = pf_clean[col].replace(-1, np.nan)
print('\n→ pf_clean: sentinel -1을 NaN으로 치환')

### 1-3. 이상치 (IQR 기준)

In [ ]:
outlier_cols = ['price_krw', 'volume_ml', 'text_length',
                'aggregate_rating_count', 'image_count']

rows_o = []
for col in outlier_cols:
    s = pf_clean[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((s < lo) | (s > hi)).sum()
    rows_o.append({'column': col, 'min': s.min(), 'median': s.median(), 'max': s.max(),
                   'IQR_lower': round(lo, 1), 'IQR_upper': round(hi, 1), 'outlier_count': n_out})

out_df = pd.DataFrame(rows_o).set_index('column').round(1)
display(out_df)

# 가격 이상치 목록
hi_price_thr = out_df.loc['price_krw', 'IQR_upper']
hi_price = pf_clean[pf_clean['price_krw'] > hi_price_thr]
print(f'\n가격 이상치 {len(hi_price)}개 (>{hi_price_thr:.0f}원):')
display(hi_price[['brand_ko', 'price_krw', 'volume_ml']].sort_values('price_krw', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['price_krw', 'volume_ml', 'text_length']):
    data = pf_clean[col].dropna()
    ax.hist(data, bins=30, edgecolor='white', color='steelblue')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('count')

plt.suptitle('주요 수치형 feature 분포', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 1-4. 중복 상품 확인

In [ ]:
url_dupes = pf['url'].duplicated().sum()
print(f'URL 중복: {url_dupes}개')

brand_counts = pf['brand_ko'].value_counts()
multi_brand  = brand_counts[brand_counts > 1]
print(f'\n브랜드당 여러 상품 보유: {len(multi_brand)}개 브랜드, 총 {multi_brand.sum()}개 상품')
print('브랜드당 상품 수 분포 (상품 수 → 해당 브랜드 수):')
display(brand_counts.value_counts().sort_index().rename('브랜드 수').to_frame().T)

---
## 2. response.csv 품질 확인

In [ ]:
print('=== 기본 구조 ===')
print(f'총 rows       : {len(resp):,}')
print(f'unique set_id : {resp["set_id"].nunique():,}')
print(f'set_id당 rows : {resp.groupby("set_id").size().value_counts().to_dict()}')
print(f'engine 분포   : {resp["engine"].value_counts().to_dict()}')
print(f'round 분포    : {resp["round"].value_counts().to_dict()}')

In [ ]:
print('=== Ranking 결측 ===')
for col in ['Y2_top1_brand', 'rank2_brand', 'rank3_brand']:
    n = resp[col].isna().sum()
    print(f'{col:22s}: {n:4d}건  ({n/len(resp)*100:.2f}%)')

no_rank3 = resp[resp['rank3_brand'].isna()]
print(f'\nrank3 누락 {len(no_rank3)}건 - engine별:', no_rank3['engine'].value_counts().to_dict())
print('→ 불완전 trial은 모델 학습 시 2-item ranking으로 처리하거나 제외 필요')

In [ ]:
print('=== 브랜드 overlap ===')
resp_brands = (set(resp['input_sku1'].dropna())
               | set(resp['input_sku2'].dropna())
               | set(resp['input_sku3'].dropna()))
pf_brands   = set(pf['brand_ko'].dropna())
print(f'response 등장 브랜드  : {len(resp_brands)}')
print(f'page_features 브랜드  : {len(pf_brands)}')
print(f'overlap               : {len(resp_brands & pf_brands)}')
only_resp = resp_brands - pf_brands
print(f'response에만 있는 브랜드 ({len(only_resp)}개): {sorted(only_resp)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
resp['response_chars'].hist(bins=40, ax=ax, edgecolor='white', color='coral')
ax.set_title('AI 응답 길이 분포 (response_chars)')
ax.set_xlabel('characters')
ax.set_ylabel('count')
plt.tight_layout(); plt.show()
print(resp['response_chars'].describe().round(1))

---
## 3. 품질 요약 및 처리 방침

In [ ]:
summary = {
    '이슈': [
        '_unused_58, _unused_59',
        'ph_value',
        'aggregate_rating_value/count',
        'volume_ml sentinel',
        'skin_type_targets (20.8%)',
        'rank3_brand 누락 1.37%',
        'brand_ko 비유일성',
    ],
    '내용': [
        '100% 결측 → 제거 완료',
        '98.2% -1 (sentinel) → 모델링 시 drop 권장',
        '38.8% -1 (sentinel) → NaN 처리 후 has_rating binary feature 추가',
        '-1이 sentinel로 혼재 → NaN 치환 필요',
        '20.8% 결측 → list 파싱 필요, count feature 활용',
        'AI가 3개 ranking 미제공 → 2-item ranking 처리 또는 제외',
        '브랜드당 최대 20개 상품 → response ↔ page_features 매핑 방식 결정 필요',
    ],
    '모델링 처리': [
        'drop',
        'binary has_ph_info',
        'NaN + has_rating indicator',
        'NaN impute or binary has_volume',
        'keep skin_type_targets_count',
        '불완전 trial 제외 (82건)',
        'brand 집계 평균 or URL 매핑 추가 수집',
    ]
}
display(pd.DataFrame(summary).set_index('이슈'))

---
## 4. Train / Validation / Test Split

### 분할 전략
| 단계 | 방식 | 용도 |
|------|------|------|
| **Step 1: trial-level split** | set_id 70 / 15 / 15 무작위 분할 | Seen-item ranking 성능 평가 |
| **Step 2: unseen-item holdout** | 브랜드 10% hold-out → 해당 set_id 전부 unseen_test | 신규 상품 일반화 성능 평가 |

분할 단위: **`set_id`** (trial). 동일 set_id가 여러 split에 걸치면 데이터 누수 발생.

In [ ]:
SEED = 42
rng  = np.random.default_rng(SEED)

set_ids  = resp['set_id'].unique()
shuffled = rng.permutation(set_ids)
n = len(shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_ids = set(shuffled[:n_train])
val_ids   = set(shuffled[n_train:n_train+n_val])
test_ids  = set(shuffled[n_train+n_val:])

print(f'전체 unique set_id: {n:,}')
print(f'  Train  : {len(train_ids):,}  ({len(train_ids)/n*100:.1f}%)')
print(f'  Val    : {len(val_ids):,}  ({len(val_ids)/n*100:.1f}%)')
print(f'  Test   : {len(test_ids):,}  ({len(test_ids)/n*100:.1f}%)')

In [ ]:
# ── Step 2: unseen-item holdout ──────────────────────────────────────────
all_brands = list(
    set(resp['input_sku1'].dropna())
    | set(resp['input_sku2'].dropna())
    | set(resp['input_sku3'].dropna())
)
rng2 = np.random.default_rng(SEED + 1)
n_holdout     = max(1, int(len(all_brands) * 0.10))
holdout_brands = set(rng2.choice(all_brands, size=n_holdout, replace=False))

print(f'전체 브랜드: {len(all_brands)}')
print(f'Holdout 브랜드 ({n_holdout}개): {sorted(holdout_brands)}')

def contains_holdout(row):
    return any(row[c] in holdout_brands
               for c in ['input_sku1', 'input_sku2', 'input_sku3']
               if pd.notna(row[c]))

resp_temp            = resp.copy()
resp_temp['_hold']   = resp_temp.apply(contains_holdout, axis=1)
unseen_ids           = set(resp_temp[resp_temp['_hold']]['set_id'])

train_ids_clean = train_ids - unseen_ids
val_ids_clean   = val_ids   - unseen_ids
test_seen_ids   = test_ids  - unseen_ids

print(f'\nunseen_test set_ids : {len(unseen_ids):,}')
print(f'train (clean)       : {len(train_ids_clean):,}')
print(f'val   (clean)       : {len(val_ids_clean):,}')
print(f'test  (seen)        : {len(test_seen_ids):,}')

In [ ]:
def assign_split(sid):
    if sid in unseen_ids:      return 'unseen_test'
    if sid in train_ids_clean: return 'train'
    if sid in val_ids_clean:   return 'val'
    if sid in test_seen_ids:   return 'test'
    return 'unassigned'

resp['split'] = resp['set_id'].map(assign_split)

print('Split 분포 (rows):')
display(resp['split'].value_counts().rename('rows').to_frame())
print()
print('split별 set_id 수:')
display(resp.groupby('split')['set_id'].nunique().rename('set_ids').to_frame())

### 4-1. Split 균형 확인 (engine / round 비율)

In [ ]:
print(f'{"split":<15} {"engine: openai":>16} {"anthropic":>12}  |  {"round1":>8} {"round2":>8}')
print('-' * 65)
for split_name, grp in resp.groupby('split'):
    eng = grp['engine'].value_counts(normalize=True)
    rnd = grp['round'].value_counts(normalize=True)
    print(f'{split_name:<15} '
          f'{eng.get("openai",0):>16.1%} {eng.get("anthropic",0):>12.1%}  |  '
          f'{rnd.get("round1",0):>8.1%} {rnd.get("round2",0):>8.1%}')

### 4-2. 저장

In [ ]:
os.makedirs(SPLIT_DIR, exist_ok=True)

for split_name in ['train', 'val', 'test', 'unseen_test']:
    subset   = resp[resp['split'] == split_name].drop(columns=['split']).reset_index(drop=True)
    out_path = f'{SPLIT_DIR}/{split_name}.csv'
    subset.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'Saved {out_path}  ({len(subset):,} rows)')

meta = {
    'seed'           : SEED,
    'split_ratio'    : '70/15/15',
    'split_unit'     : 'set_id',
    'holdout_brands' : sorted(holdout_brands),
    'set_id_counts'  : {
        'train'      : len(train_ids_clean),
        'val'        : len(val_ids_clean),
        'test'       : len(test_seen_ids),
        'unseen_test': len(unseen_ids),
    },
    'row_counts': {s: int((resp['split'] == s).sum())
                   for s in ['train', 'val', 'test', 'unseen_test']},
}
with open(f'{SPLIT_DIR}/split_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('\nSaved split_meta.json')

---
## 5. 최종 요약

In [ ]:
print('=== EDA 품질 이슈 요약 ===')
issues = [
    ('_unused 컬럼 2개',           '100% 결측 → 제거 완료'),
    ('ph_value',                   '98.2% sentinel(-1) → 모델에서 drop 권장'),
    ('aggregate_rating_value/count','38.8% sentinel(-1) → NaN + binary indicator'),
    ('rank3_brand 누락',            f'1.37% ({resp["rank3_brand"].isna().sum()}건) → 처리 방침 결정 필요'),
    ('brand_ko 비유일성',           '122개 브랜드, 327개 상품 → 매핑 방식 결정 필요'),
]
for issue, note in issues:
    print(f'  [{issue}] {note}')

print()
print('=== Split 결과 ===')
split_counts  = resp['split'].value_counts()
split_setids  = resp.groupby('split')['set_id'].nunique()
for s in ['train', 'val', 'test', 'unseen_test']:
    print(f'  {s:<15}: {split_counts.get(s, 0):5,} rows  /  {split_setids.get(s, 0):4,} set_ids')

---
## 5. full_prompt.csv 기반 URL 매핑 및 page_features 병합

### 배경
`full_prompt.csv`는 각 set_id에 대해 sku1/2/3의 정확한 URL을 포함한다.
URL로 page_features를 1:1 매핑하면 브랜드 비유일성 문제가 해소된다.

### URL 상태 분류
| 상태 | 건수 | 처리 방법 |
|------|------|-----------|
| 정확한 일치 (len < 100) | 218개 URL | direct join |
| 100자 truncated → prefix 1:1 복원 | 77개 URL | prefix match |
| 100자 truncated → prefix ambiguous | 2개 URL (180 trials) | brand-level 평균 feature |

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_DIR  = '../data/raw'
PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)

fp   = pd.read_csv(f'{RAW_DIR}/full_prompt.csv', encoding='utf-8')
pf   = pd.read_csv(f'{RAW_DIR}/page_features.csv', encoding='utf-8')
resp = pd.read_csv(f'{RAW_DIR}/response.csv', encoding='utf-8')

pf = pf.drop(columns=[c for c in pf.columns if pf[c].isna().all()])
pf_urls = list(pf['url'])

# ── URL → 정규화된 URL 매핑 테이블 구축 ──────────────────────────────────
# 모든 fp URL 수집
all_fp_urls = set()
for i in [1, 2, 3]:
    all_fp_urls |= set(fp[f'sku{i}_url'].dropna())

url_resolve = {}   # fp_url → pf_url (or None if ambiguous)
url_is_ambig = {}  # fp_url → True/False

for fu in all_fp_urls:
    if fu in set(pf_urls):
        url_resolve[fu]   = fu
        url_is_ambig[fu] = False
    elif len(fu) == 100:
        matches = [pu for pu in pf_urls if pu.startswith(fu)]
        if len(matches) == 1:
            url_resolve[fu]   = matches[0]
            url_is_ambig[fu] = False
        else:
            url_resolve[fu]   = None   # ambiguous
            url_is_ambig[fu] = True
    else:
        url_resolve[fu]   = None
        url_is_ambig[fu] = False

resolved  = sum(1 for v in url_resolve.values() if v is not None)
ambiguous = sum(1 for v in url_is_ambig.values() if v)
unresolved = sum(1 for v in url_resolve.values() if v is None and not url_is_ambig.get(v, False))

print(f'전체 unique fp_url   : {len(all_fp_urls)}')
print(f'  정확히 해소됨       : {resolved}')
print(f'  ambiguous (brand avg): {ambiguous}')
print(f'  기타 미매핑          : {unresolved - ambiguous}')


In [ ]:
# Sentinel -1 → NaN
SENTINEL_COLS = ['ph_value', 'aggregate_rating_value', 'aggregate_rating_count', 'volume_ml']
pf_clean = pf.copy()
for col in SENTINEL_COLS:
    if col in pf_clean.columns:
        pf_clean[col] = pf_clean[col].replace(-1, np.nan)

# numeric feature 컬럼 (모델에 사용 가능한 것)
FEATURE_COLS = [c for c in pf_clean.select_dtypes(include='number').columns]
ID_COLS = ['brand_ko', 'url']

# brand-level 평균 (ambiguous URL 처리 및 일반 분석에 활용)
brand_agg = (pf_clean
             .groupby('brand_ko')[FEATURE_COLS]
             .mean()
             .reset_index())
brand_agg.columns = ['brand_ko'] + [f'{c}__brand_avg' for c in FEATURE_COLS]

print(f'brand_agg shape: {brand_agg.shape}')
print(f'브랜드 수: {len(brand_agg)}')


In [ ]:
# 각 trial의 3개 상품을 long format으로 전개
# 결과: (set_id, sku_pos) → url_resolved, brand_ko, is_ambiguous

rows = []
for _, row in fp.iterrows():
    sid = row['set_id']
    for pos in [1, 2, 3]:
        fu    = row[f'sku{pos}_url']
        brand = row[f'sku{pos}_brand']
        res   = url_resolve.get(fu)
        ambig = url_is_ambig.get(fu, False)
        rows.append({
            'set_id'      : sid,
            'sku_pos'     : pos,
            'brand_ko'    : brand,
            'fp_url'      : fu,
            'resolved_url': res,
            'is_ambiguous': ambig,
        })

trial_items = pd.DataFrame(rows)

# page_features 병합 (exact match)
pf_indexed = pf_clean.set_index('url')
exact_mask  = trial_items['resolved_url'].notna() & ~trial_items['is_ambiguous']
ambig_mask  = trial_items['is_ambiguous']

# exact match rows → pf row 붙이기
exact_rows = trial_items[exact_mask].copy()
pf_feat = pf_indexed[FEATURE_COLS].rename(columns={c: c for c in FEATURE_COLS})
exact_rows = exact_rows.join(pf_feat, on='resolved_url')

# ambiguous rows → brand avg 붙이기
ambig_rows = trial_items[ambig_mask].copy()
ambig_rows = ambig_rows.merge(brand_agg, on='brand_ko', how='left')
# brand_avg 컬럼명을 FEATURE_COLS와 동일하게 rename
rename_map = {f'{c}__brand_avg': c for c in FEATURE_COLS}
ambig_rows = ambig_rows.rename(columns=rename_map)

# 나머지 행 (매핑 불가 - 없어야 정상)
other_rows = trial_items[~exact_mask & ~ambig_mask].copy()

# 합치기
trial_items_full = pd.concat([exact_rows, ambig_rows, other_rows], ignore_index=True)
trial_items_full = trial_items_full.sort_values(['set_id', 'sku_pos']).reset_index(drop=True)

print(f'trial_items_full shape: {trial_items_full.shape}')
print(f'  exact match rows  : {len(exact_rows)}')
print(f'  brand avg rows    : {len(ambig_rows)}')
print(f'  unmapped rows     : {len(other_rows)}')
print(f'  feature NaN rate  : {trial_items_full[FEATURE_COLS].isna().mean().mean():.2%}')


In [ ]:
# response.csv에서 각 (set_id, sku_pos)의 AI ranking 추출
# input_sku1/2/3 → sku_pos 1/2/3 순서는 full_prompt와 동일 (brand_match 100% 확인됨)

# response는 set_id당 2행 (engine별). 여기선 ranking만 추출하므로 wide → long
resp_rank_rows = []
for _, row in resp.iterrows():
    sid    = row['set_id']
    engine = row['engine']
    rnd    = row['round']
    # 입력 순서 확인
    sku_to_pos = {row[f'input_sku{i}']: i for i in [1, 2, 3]}
    # ranking: top1 > rank2 > rank3
    ranking = {}
    if pd.notna(row['Y2_top1_brand']): ranking[row['Y2_top1_brand']] = 1
    if pd.notna(row['rank2_brand']):   ranking[row['rank2_brand']]   = 2
    if pd.notna(row['rank3_brand']):   ranking[row['rank3_brand']]   = 3

    for brand, ai_rank in ranking.items():
        pos = sku_to_pos.get(brand)
        if pos:
            resp_rank_rows.append({
                'set_id'  : sid,
                'engine'  : engine,
                'round'   : rnd,
                'sku_pos' : pos,
                'brand_ko': brand,
                'ai_rank' : ai_rank,
            })

resp_long = pd.DataFrame(resp_rank_rows)
print(f'resp_long shape: {resp_long.shape}')
print(f'ai_rank 분포: {resp_long["ai_rank"].value_counts().to_dict()}')
print(f'NaN ai_rank (rank3 누락 등): {resp_long["ai_rank"].isna().sum()}')


In [ ]:
# trial_items_full + resp_long 병합 → 최종 학습 데이터
# key: (set_id, engine, round, sku_pos)
# 결과: 각 (trial, engine, round)에서 3개 상품 × page_features + ai_rank

train_data = resp_long.merge(
    trial_items_full[['set_id', 'sku_pos', 'brand_ko', 'resolved_url', 'is_ambiguous'] + FEATURE_COLS],
    on=['set_id', 'sku_pos', 'brand_ko'],
    how='left'
)

print(f'최종 학습 데이터 shape: {train_data.shape}')
print(f'feature NaN rate: {train_data[FEATURE_COLS].isna().mean().mean():.2%}')
print(f'is_ambiguous 비율: {train_data["is_ambiguous"].mean():.2%}')
print()
print('ai_rank별 행 수:')
print(train_data['ai_rank'].value_counts().sort_index())


In [ ]:
# 저장
trial_items_full.to_csv(f'{PROC_DIR}/trial_items.csv', index=False, encoding='utf-8-sig')
train_data.to_csv(f'{PROC_DIR}/train_data_long.csv', index=False, encoding='utf-8-sig')

print(f'Saved: {PROC_DIR}/trial_items.csv       ({len(trial_items_full):,} rows)')
print(f'Saved: {PROC_DIR}/train_data_long.csv   ({len(train_data):,} rows)')
print()
print('컬럼 구성:')
print(f'  ID cols     : set_id, engine, round, sku_pos, brand_ko, resolved_url, is_ambiguous')
print(f'  Feature cols: {len(FEATURE_COLS)}개 numeric features from page_features')
print(f'  Target      : ai_rank (1=best, 3=worst)')


### 5-1. 기존 split에 매핑 데이터 적용

In [ ]:
SPLIT_DIR = '../data/splits'

for split_name in ['train', 'val', 'test', 'unseen_test']:
    split_ids = pd.read_csv(f'{SPLIT_DIR}/{split_name}.csv',
                            encoding='utf-8-sig', usecols=['set_id'])['set_id'].unique()
    subset = train_data[train_data['set_id'].isin(split_ids)].reset_index(drop=True)
    out_path = f'{PROC_DIR}/{split_name}_features.csv'
    subset.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'{split_name:15s}: {len(subset):6,} rows → {out_path}')


### 5-2. 매핑 결과 요약

In [ ]:
print('=== URL 매핑 결과 ===')
print(f'  exact match   : {len(exact_rows):,}건  (url로 1:1 매핑)')
print(f'  prefix match  : {sum(1 for v in url_is_ambig.values() if not v and url_resolve.get(list(all_fp_urls)[0]) is not None):,}건  (100자 잘린 URL 복원)')
print(f'  brand avg     : {len(ambig_rows):,}건  (2개 ambiguous URL → brand 평균 feature)')
print(f'  unmapped      : {len(other_rows):,}건')
print()
print('=== 최종 학습 데이터 ===')
print(f'  전체 rows     : {len(train_data):,}')
print(f'  feature cols  : {len(FEATURE_COLS)}개')
print(f'  engines       : {sorted(train_data["engine"].unique())}')
print(f'  rounds        : {sorted(train_data["round"].unique())}')
print(f'  ai_rank range : {train_data["ai_rank"].min()} ~ {train_data["ai_rank"].max()}')
